# Lexicon baselines

Two rule-based sentiment classifiers evaluated on the held-out test set. Neither trains: both only read the test split.

1. **VADER** — compound score with $\pm 0.05$ thresholds on raw text.
2. **SentiWordNet** — POS-aware WordNet polarity averaged over content tokens.

Produces a shared summary table at the end so the two are directly comparable.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
import spacy

from sklearn.metrics import classification_report, f1_score

from utils import normalize_text

nltk.download('vader_lexicon')
for pkg in ['wordnet', 'sentiwordnet', 'omw-1.4']:
    try:
        nltk.data.find(f'corpora/{pkg}')
    except LookupError:
        nltk.download(pkg)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load the SAME test set used by Naive Bayes and RoBERTa for apples-to-apples comparison
lex_df = pd.read_csv('../data/test_data.csv')
lex_df = lex_df.dropna(subset=['appName', 'content', 'Sentiment']).copy()
lex_df = lex_df[lex_df['content'].astype(str).str.strip() != ''].reset_index(drop=True)

# Keep raw content for VADER; build a normalized variant for SentiWordNet
# (whitespace collapsed, newlines stripped) so tokenization and POS-tagging behave predictably.
lex_df['content_norm'] = lex_df['content'].apply(normalize_text)

y_test = lex_df['Sentiment']
print('Rows:', len(lex_df))
print('Sentiment counts:', y_test.value_counts().to_dict())

Rows: 24438
Sentiment counts: {'Positive': 18885, 'Negative': 4361, 'Neutral': 1192}


## 1. VADER

VADER uses punctuation and capitalisation to score sentiment — apply to RAW text.

In [3]:
vader = SentimentIntensityAnalyzer()

def vader_prediction(text):
    compound = vader.polarity_scores(text)['compound']
    if compound >= 0.05:
        return 'Positive'
    if compound <= -0.05:
        return 'Negative'
    return 'Neutral'

lex_df['vader_pred'] = lex_df['content'].apply(vader_prediction)
f1_vader = f1_score(y_test, lex_df['vader_pred'], average='macro')
print(f'VADER macro F1: {f1_vader:.4f}')
print(classification_report(y_test, lex_df['vader_pred']))

VADER macro F1: 0.4919
              precision    recall  f1-score   support

    Negative       0.71      0.45      0.55      4361
     Neutral       0.06      0.29      0.10      1192
    Positive       0.90      0.76      0.82     18885

    accuracy                           0.68     24438
   macro avg       0.56      0.50      0.49     24438
weighted avg       0.82      0.68      0.74     24438



## 2. SentiWordNet

Scores lemmatized non-stopword tokens using the first synset per POS, then averages positive minus negative polarity. Operates on the normalized text variant.

In [4]:
nlp = spacy.load('en_core_web_sm')

WN_POS = {
    'NOUN': wn.NOUN,
    'VERB': wn.VERB,
    'ADJ': wn.ADJ,
    'ADV': wn.ADV,
}

def sentiwordnet_score(text: str) -> float:
    doc = nlp(text)
    score = 0.0
    hits = 0
    for t in doc:
        if t.is_stop or t.is_punct or t.is_space:
            continue
        wn_pos = WN_POS.get(t.pos_)
        if wn_pos is None:
            continue
        synsets = wn.synsets(t.lemma_, pos=wn_pos)
        if not synsets:
            continue
        try:
            swn_syn = swn.senti_synset(synsets[0].name())
        except Exception:
            continue
        score += (swn_syn.pos_score() - swn_syn.neg_score())
        hits += 1
    return score / hits if hits else 0.0

def score_to_label(s, pos_thr=0.05, neg_thr=-0.05):
    if s >= pos_thr:
        return 'Positive'
    if s <= neg_thr:
        return 'Negative'
    return 'Neutral'

lex_df['swn_score'] = lex_df['content_norm'].apply(sentiwordnet_score)
lex_df['swn_pred'] = lex_df['swn_score'].apply(score_to_label)

f1_swn = f1_score(y_test, lex_df['swn_pred'], average='macro')
print(f'SentiWordNet macro F1: {f1_swn:.4f}')
print(classification_report(y_test, lex_df['swn_pred']))

SentiWordNet macro F1: 0.4111
              precision    recall  f1-score   support

    Negative       0.60      0.29      0.39      4361
     Neutral       0.06      0.44      0.10      1192
    Positive       0.89      0.63      0.74     18885

    accuracy                           0.56     24438
   macro avg       0.52      0.45      0.41     24438
weighted avg       0.80      0.56      0.65     24438



## 3. Summary

In [5]:
summary = pd.DataFrame([
    {'system': 'VADER',        'macro_f1': round(f1_vader, 4)},
    {'system': 'SentiWordNet', 'macro_f1': round(f1_swn, 4)},
])
print(summary.to_string(index=False))

      system  macro_f1
       VADER    0.4919
SentiWordNet    0.4111
